# Setup

In [ ]:
# 1. Install dependencies
%conda install python=3.12
%pip install --quiet pydantic-ai sentence-transformers numpy mcp nest_asyncio openai
# 1b. Patch asyncio for Jupyter (REQUIRED — run this before any agent call)
import nest_asyncio
nest_asyncio.apply()


In [ ]:
# 2. Set your OpenRouter API key
# (Get one at https://openrouter.ai — gives access to 300+ models with one key.)
#
# Paste your key here, OR leave it as None to be prompted interactively,
# OR set it in your shell as the env var OPENROUTER_API_KEY.
OPENROUTER_API_KEY = ''    # e.g. 'sk-or-v1-...'

import os, getpass

if OPENROUTER_API_KEY:
    os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEY
elif 'OPENROUTER_API_KEY' not in os.environ:
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter API key: ')

In [3]:
import httpx
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai import Agent
from pydantic import BaseModel

MODEL = OpenAIChatModel(
    'anthropic/claude-haiku-4.5',
    provider=OpenAIProvider(
        base_url='https://openrouter.ai/api/v1',
        api_key=os.environ['OPENROUTER_API_KEY'],
        http_client=httpx.AsyncClient(timeout=60),
    ),
)

# Multiple agents

In [4]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent

class PaperSummary(BaseModel):
    authors: list[str] | None = Field(description="Name and Surname of the author, None if not available")
    method: str = Field(description="The method used in the paper")
    dataset: str = Field(description="The dataset used in the paper")
    problem: str = Field(description="The problem the paper is solving")
    achieved_accuracy: float | None = Field(description="The accuracy achieved by the method, None if not available")

extractor = Agent(
    MODEL,
    output_type=PaperSummary,
    system_prompt='Extract structured fields from a paper abstract.',
)

summariser = Agent(
    MODEL,
    system_prompt=(
        'You are given structured fields about a paper. '
        'Write ONE plain-English sentence summarising the work.'
    ),
)

def process_abstract(text: str) -> str:
    extractor_output = extractor.run_sync(text).output
    summary = summariser.run_sync(str(extractor_output)).output
    return summary

abstract =(
    "We developed a graph neural network for molecular property prediction "
    "using atom-level message passing and bond-aware attention. Unlike the "
    "image-classification studies, this work did not evaluate on ImageNet-1K, "
    "CIFAR-100, or CIFAR-10 because its task involved molecular graphs rather "
    "than natural images. Instead, the model was evaluated on MoleculeNet "
    "benchmarks including Tox21, ClinTox, and HIV. It achieved 78% accuracy "
    "on Tox21, 74% accuracy on ClinTox, and 81% accuracy on HIV. The model "
    "also reached 86% AUROC on Tox21 and 91% AUROC on HIV, indicating strong "
    "performance on binary molecular classification tasks."
)
process_abstract(abstract)

'This work uses a graph neural network with atom-level message passing and bond-aware attention mechanisms to predict molecular properties, achieving 81% accuracy on standard benchmarks like Tox21, ClinTox, and HIV.'

# Dependencies

In [5]:
# We can use deps to bring more control to the multi-agent workflows
from pydantic_ai import RunContext

extractor = Agent(
    MODEL,
    output_type=PaperSummary,
    system_prompt="Extract structured fields from a paper abstract.",
)

summariser = Agent(
    MODEL,
    deps_type=PaperSummary,
    system_prompt=(
        "You are given structured fields about a paper. "
        "Write ONE plain-English sentence summarising the work. "
        "Use only the information provided in the dependencies."
    ),
)

@summariser.system_prompt
def add_paper_summary(ctx: RunContext[PaperSummary]) -> str:
    paper: PaperSummary = ctx.deps

    # We can omit non-relevant fields like authors
    return (
        f"- Method: {paper.method}\n"
        f"- Dataset: {paper.dataset}\n"
        f"- Problem: {paper.problem}\n"
        f"- Achieved accuracy: {paper.achieved_accuracy}\n"
    )

def process_abstract_with_deps(text: str) -> str:
    extractor_output = extractor.run_sync(text).output
    summary = summariser.run_sync(deps=extractor_output).output
    return summary


abstract = (
    "We developed a graph neural network for molecular property prediction "
    "using atom-level message passing and bond-aware attention. Unlike the "
    "image-classification studies, this work did not evaluate on ImageNet-1K, "
    "CIFAR-100, or CIFAR-10 because its task involved molecular graphs rather "
    "than natural images. Instead, the model was evaluated on MoleculeNet "
    "benchmarks including Tox21, ClinTox, and HIV. It achieved 78% accuracy "
    "on Tox21, 74% accuracy on ClinTox, and 81% accuracy on HIV. The model "
    "also reached 86% AUROC on Tox21 and 91% AUROC on HIV, indicating strong "
    "performance on binary molecular classification tasks."
)

process_abstract_with_deps(abstract)

'A graph neural network with atom-level message passing and bond-aware attention achieves 0.81 accuracy on molecular property prediction tasks across MoleculeNet benchmarks including Tox21, ClinTox, and HIV datasets.'

# Exercise

In [ ]:
# Create an agentic workflow that will create a script called draw.py that will make ascii art of a requested animal
# You should guarantee the script will take a parameter --color
# Bonus - create a judge agent that will evaluate the quality of the art, and will have configurable (promptable) style preferences

In [ ]:
import subprocess
import tempfile

def write_python_script(code: str) -> str:
    """
    Write Python code to a temporary .py file and return the file path.
    """
    with tempfile.NamedTemporaryFile(
        suffix=".py",
        mode="w",
        delete=False,
        encoding="utf-8",
    ) as f:
        f.write(code)
        return f.name


def run_python_script(script_path: str, args: list[str] | None = None) -> str:
    """
    Run a Python script with optional command-line arguments.

    Example:
        run_python_script(script_path, ["--color", "red"])
    """
    if args is None:
        args = []

    proc = subprocess.run(
        ["python3", script_path, *args],
        capture_output=True,
        text=True,
        timeout=10,
    )

    output = proc.stdout + proc.stderr
    return output.strip() or "(no output)"

In [ ]:
# Test 
print(run_python_script('./draw.py', args=["--color", "red"]))


 /\_/\
( o.o )
 > ^ <

